## Описание проекта

Интернет-магазин собирает историю покупателей, проводит рассылки предложений и планирует будущие продажи. Для оптимизации процессов надо выделить пользователей, которые готовы совершить покупку в ближайшее время.

### Цель

Предсказать вероятность покупки в течение 90 дней

### Задачи

* Изучить данные
* Разработать полезные признаки
* Создать модель для классификации пользователей
* Улучшить модель и максимизировать метрику roc_auc
* Выполнить тестирование

### Данные

`apparel-purchases`

история покупок
* `client_id` - идентификатор пользователя
* `quantity` - количество товаров в заказе
* `price` цена товара
* `category_ids` - вложенные категории, к которым отнсится товар
* `date` - дата покупки
* `message_id` - идентификатор сообщения из рассылки

`apparel-target_binary`

совершит ли клиент покупку в течение следующих 90 дней
* `client_id` - идентификатор пользователя
* `target` - целевой признак

### Результат

Репозиторий на гитхабе:
* тетрадь jupyter notebook с описанием, подготовкой признаков, обучением модели и тестированием
* описание проекта и инструкция по использованию в файле README.md
* список зависимостей в файле requirements.txt

### Стэк

* python
* pandas
* sklearn

## Описание данных

### `apparel-purchases`

Данные о покупках клиентов по дням и по товарам. В каждой записи покупка определенного товара, его цена, количество штук.

В таблице есть списки идентификаторов, к каким категориям относится товар. Часто это вложенные категории (например автотовары-аксессуары-освежители), но также может включать в начале списка маркер распродажи или маркер женщинам/мужчинам.

Нумерация категорий сквозная для всех уровней, то есть 44 на второй позиции списка или на третьей – это одна и та же категория. Иногда дерево категорий обновляется, поэтому могут меняться вложенности, например `['4', '28', '44', '1594']` или `['4', '44', '1594']`. Как обработать такие случаи – можете предлагать свои варианты решения.
* `client_id` - идентификатор клиента
* `quantity` - количество единиц товара
* `price` - цена товара
* `category_ids` - идентификаторы категорий
* `date` - дата покупки
* `message_id` - идентификатор сообщения из рассылки

### `apparel-messages`

Рассылки, которые были отправлены клиентам из таблицы покупок.
* `bulk_campaign_id` - идентификатор рассылки
* `client_id` - идентификатор клиента
* `message_id` - идентификатор сообщения
* `event` - действие с сообщением (отправлено, открыто, покупка...)
* `channel` - канал рассылки
* `date` - дата действия
* `created_at` - дата-время полностью

### `target`

* `client_id` - идентификатор клиента
* `target` - клиент совершил покупку в целевом периоде

Общая база рассылок огромна, поэтому собрали для вас агрегированную по дням статистику по рассылкам. Если будете создавать на основе этой статистики дополнительные признаки, обратите внимание, что нельзя суммировать по колонкам nunique, потому что это уникальные клиенты в пределах дня, у вас нет данных, повторяются ли они в другие дни.

### `full_campaign_daily_event`

Агрегация общей базы рассылок по дням и типам событий
* `date` - дата
* `bulk_campaign_id` - идентификатор рассылки
* `count_event*` - общее количество каждого события event
* `nunique_event*` - количество уникальных client_id в каждом событии

\* в именах колонок найдете все типы событий event

### `full_campaign_daily_event_channel`

Агрегация по дням с учетом событий и каналов рассылки
* `date` - дата
* `bulk_campaign_id` - идентификатор рассылки
* `count_event*_channel*` - общее количество каждого события по каналам
* `nunique_event*_channel*` - количество уникальных client_id по событиям и каналам

\* в именах колонок есть все типы событий event и каналов рассылки channel

## Импорты библиотек

In [ ]:
import pandas as pd 
import numpy as np 

from tqdm import tqdm
tqdm.pandas()

from joblib import Parallel, delayed
import time
from prettytable import PrettyTable

Функция для красивого вывода

In [110]:
def show_html(text, color="#1D2227"):
    from IPython.display import HTML, display
    display(HTML(f'<span style="background-color: {color};">{text}</span>'))

Загрузка файлов и вывод первых строк каждого файла

In [111]:
file_names = ['apparel-purchases.csv', 'apparel-messages.csv', 'apparel-target_binary.csv', \
    'full_campaign_daily_event.csv', 'full_campaign_daily_event_channel.csv']

dataframes = []

for i, file_name in enumerate(file_names):
    dataframe = pd.read_csv('./Данные/' + file_name)
    dataframes.append(dataframe)
    show_html(file_name)
    print(dataframe.shape)
    if i < 3:
        display(dataframe.head())
    else:
        display(dataframe.head().T)

(202208, 6)


,client_id,quantity,price,category_ids,date,message_id
0,1515915625468169594,1,1999.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
1,1515915625468169594,1,2499.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
2,1515915625471138230,1,6499.0,"['4', '28', '57', '431']",2022-05-16,1515915625471138230-4437-6282242f27843
3,1515915625471138230,1,4999.0,"['4', '28', '244', '432']",2022-05-16,1515915625471138230-4437-6282242f27843
4,1515915625471138230,1,4999.0,"['4', '28', '49', '413']",2022-05-16,1515915625471138230-4437-6282242f27843


(12739798, 7)


,bulk_campaign_id,client_id,message_id,event,channel,date,created_at
0,4439,1515915625626736623,1515915625626736623-4439-6283415ac07ea,open,email,2022-05-19,2022-05-19 00:14:20
1,4439,1515915625490086521,1515915625490086521-4439-62834150016dd,open,email,2022-05-19,2022-05-19 00:39:34
2,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,open,email,2022-05-19,2022-05-19 00:51:49
3,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,click,email,2022-05-19,2022-05-19 00:52:20
4,4439,1515915625471518311,1515915625471518311-4439-628341570c133,open,email,2022-05-19,2022-05-19 00:56:52


(49849, 2)


,client_id,target
0,1515915625468060902,0
1,1515915625468061003,1
2,1515915625468061099,0
3,1515915625468061100,0
4,1515915625468061170,0


(131072, 24)


,0,1,2,3,4
date,2022-05-19,2022-05-19,2022-05-19,2022-05-19,2022-05-19
bulk_campaign_id,563,577,622,634,676
count_click,0,0,0,0,0
count_complain,0,0,0,0,0
count_hard_bounce,0,0,0,0,0
count_open,4,1,2,1,1
count_purchase,0,0,0,0,0
count_send,0,0,0,0,0
count_soft_bounce,0,0,0,0,0
count_subscribe,0,0,0,0,0


(131072, 36)


,0,1,2,3,4
date,2022-05-19,2022-05-19,2022-05-19,2022-05-19,2022-05-19
bulk_campaign_id,563,577,622,634,676
count_click_email,0,0,0,0,0
count_click_mobile_push,0,0,0,0,0
count_open_email,4,1,2,1,1
count_open_mobile_push,0,0,0,0,0
count_purchase_email,0,0,0,0,0
count_purchase_mobile_push,0,0,0,0,0
count_soft_bounce_email,0,0,0,0,0
count_subscribe_email,0,0,0,0,0


#### Сравнение списков `client_id`

Рассчитаем число уникальных `client_id` для всех таблиц и попарное число пересечений. 

Для таблиц `full_campaign_daily_event.csv` и `full_campaign_daily_event_channel.csv` сначала объединим их по полю `client_id` с таблицей `apparel_messages.csv` внутренним образом.

In [113]:
client_id_dict = {}
client_id_dict['apparel-purchases.csv'] = set(dataframes[0]['client_id'].dropna().unique())
client_id_dict['apparel-messages.csv'] = set(dataframes[1]['client_id'].dropna().unique())
client_id_dict['apparel-target_binary.csv'] = set(dataframes[2]['client_id'].dropna().unique())
client_id_dict['full_campaign_daily_event.csv'] = \
    set((pd.merge(dataframes[1], dataframes[3], on=['bulk_campaign_id', 'date'], how='inner'))['client_id'].dropna().unique())
client_id_dict['full_campaign_daily_event_channel.csv'] = \
    set((pd.merge(dataframes[1], dataframes[4], on=['bulk_campaign_id', 'date'], how='inner'))['client_id'].dropna().unique())

In [114]:
field_names = ["df1/df2"] + file_names
mytable = PrettyTable()
mytable.field_names = field_names

for file_name1 in client_id_dict.keys():
    data1 = client_id_dict[file_name1]
    row = []
    row.append(file_name1)
    for file_name2 in client_id_dict.keys():
        data2 = client_id_dict[file_name2]
        intersection = data1 & data2
        row.append((len(data1), len(intersection), len(data2)))
    mytable.add_row(row)
print(mytable)

+---------------------------------------+-----------------------+-----------------------+---------------------------+-------------------------------+---------------------------------------+
|                df1/df2                | apparel-purchases.csv |  apparel-messages.csv | apparel-target_binary.csv | full_campaign_daily_event.csv | full_campaign_daily_event_channel.csv |
+---------------------------------------+-----------------------+-----------------------+---------------------------+-------------------------------+---------------------------------------+
|         apparel-purchases.csv         | (49849, 49849, 49849) | (49849, 41982, 53329) |   (49849, 49849, 49849)   |     (49849, 41982, 53329)     |         (49849, 41982, 53329)         |
|          apparel-messages.csv         | (53329, 41982, 49849) | (53329, 53329, 53329) |   (53329, 41982, 49849)   |     (53329, 53329, 53329)     |         (53329, 53329, 53329)         |
|       apparel-target_binary.csv       | (49849, 

Можем сделать следующие выводы:
* Для `7 867` клиентов нет информации о рассылках - в таблице *target* `49849` уникальных `client_id`, в таблице *messages* 41982 уникальных `client_id`. Для этих же клиентов есть информация о покупках - в таблице *purchases* столько же строк, сколько и в таблице *target*
* В таблицах `apparel-messages.csv`, `full_campaign_daily_event.csv` и `full_campaign_daily_event_channel.csv` получилось одинаковой количество `client_id`, т.е. в таблицах `full_campaign_daily_event.csv` и `full_campaign_daily_event_channel.csv` информация обо всех рассылках из `apparel-messages.csv`
* Оставляем всех клиентов из файла `apparel-target_binary.csv`, но далее будет необходимо заполнить пропуски

Перед обучением модели необходимо сформировать таблицу со значениями признаков для каждого пользователя

### Варианты признаков по каждой таблице

In [115]:
def values_and_type(data):
    dict_to_df = {'column_name': [], 'type': [], 'value_1': [], 'value_2': [], 'value_3': [], 'value_4': [], 'value_5': []}
    for column in data.columns:
        dict_to_df['column_name'].append(column)
        dict_to_df['type'].append(data[column].dtype)
        values = data[column].value_counts().index.to_list()
        for i in range(5):
            if i < len(values):
                dict_to_df['value_' + str(i+1)].append(values[i])
            else:
                dict_to_df['value_' + str(i+1)].append('-')

    new_df = pd.DataFrame(data=dict_to_df)
    display(new_df)

#### 1. `apparel_purchases`

Размер таблицы и первые строки

In [116]:
apparel_purchases = dataframes[0]
print(apparel_purchases.shape)
display(apparel_purchases.head())

(202208, 6)


,client_id,quantity,price,category_ids,date,message_id
0,1515915625468169594,1,1999.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
1,1515915625468169594,1,2499.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
2,1515915625471138230,1,6499.0,"['4', '28', '57', '431']",2022-05-16,1515915625471138230-4437-6282242f27843
3,1515915625471138230,1,4999.0,"['4', '28', '244', '432']",2022-05-16,1515915625471138230-4437-6282242f27843
4,1515915625471138230,1,4999.0,"['4', '28', '49', '413']",2022-05-16,1515915625471138230-4437-6282242f27843


Проверим типы

In [117]:
values_and_type(apparel_purchases)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,client_id,int64,1515915625853312319,1515915625624308268,1515915625607113301,1515915625500232103,1515915625470860261
1,quantity,int64,1,2,3,4,6
2,price,float64,999.0,1499.0,1999.0,699.0,599.0
3,category_ids,object,"['4', '28', '57', '431']","['4', '28', '260', '420']","['4', '28', '244', '432']",[],"['4', '28', '275', '421']"
4,date,object,2022-11-11,2023-06-10,2023-04-28,2022-11-15,2022-12-22
5,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625624308268-7803-636dfea7ca890,1515915625880116527-8725-639b2f0ad5e2e,1515915625607113301-13818-650a8af70fd67,1515915625558691508-14212-6567128370bdd


Описание признаков

* `client_id` - идентификатор клиента
* `quantity` - количество единиц товара
* `price` - цена товара
* `category_ids` - идентификаторы категорий
* `date` - дата покупки
* `message_id` - идентификатор сообщения из рассылки

Варианты признаков для использования моделью

* `client_id` - нужен в итоговой таблице, но в модель подаваться не будет

* `total_quantity_period*` - общее число покупок за период. Можно рассмотреть следующие периоды - неделя, месяц, 3 месяца, полгода.

* `total_spent_period*` - общее количество потраченных средств за период. `period*` - промежуток времени, неделя, месяц, 3 месяца, полгода.

* `avg_quantity_period*` - среднее число товаров за период (можем суммарное число товаров делить на длительность периода или на число только тех дней, когда были заказы). `period*` - промежуток времени, неделя, месяц, 3 месяца, полгода.

* `avg_quantity_per_order` - среднее число товаров за заказ

* `avg_spent_period*` -  среднее число потраченных средств за период (можем сумму потраченных средств делить на длительность периода или на число только тех дней, когда были заказы). `period*` - промежуток времени, неделя, месяц, 3 месяца, полгода.

* `best_categories_count*_level_number*` - наиболее популярные категории каждого уровня у человека. Если категория встречается на разных уровнях, то считаем раздельно. `count*` - число категорий в признаке, `number*` - номер уровня.

* `days_after_last_purchase` - число дней с последней покупки


#### 2. `apparel-messages`

Размер таблицы и первые строки

In [118]:
apparel_messages = dataframes[1]
print(apparel_messages.shape)
display(apparel_messages.head())

(12739798, 7)


,bulk_campaign_id,client_id,message_id,event,channel,date,created_at
0,4439,1515915625626736623,1515915625626736623-4439-6283415ac07ea,open,email,2022-05-19,2022-05-19 00:14:20
1,4439,1515915625490086521,1515915625490086521-4439-62834150016dd,open,email,2022-05-19,2022-05-19 00:39:34
2,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,open,email,2022-05-19,2022-05-19 00:51:49
3,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,click,email,2022-05-19,2022-05-19 00:52:20
4,4439,1515915625471518311,1515915625471518311-4439-628341570c133,open,email,2022-05-19,2022-05-19 00:56:52


Посмотрим на уникальные значения в столбцах `event` и `channel`

In [119]:
display(apparel_messages['event'].unique())
display(apparel_messages['channel'].unique())

array(['open', 'click', 'purchase', 'send', 'unsubscribe', 'hbq_spam',
       'hard_bounce', 'subscribe', 'soft_bounce', 'complain', 'close'],
      dtype=object)

array(['email', 'mobile_push'], dtype=object)

Значения поля `event` в дальнейшем разобьем на позитивные, негативные и нейтральные действия

По значениям поля `channel` рассчитаем наиболее частый канал для пользователя

Описание признаков

* `bulk_campaign_id` - идентификатор рассылки
* `client_id` - идентификатор клиента
* `message_id` - идентификатор сообщения
* `event` - действие с сообщением (отправлено, открыто, покупка...)
* `channel` - канал рассылки
* `date` - дата действия
* `created_at` - дата-время полностью

Значения поля `event`
* `open` - письмо/уведомление было открыто
* `click` - пользователь перешел по ссылке в письме/уведомлении
* `purchase` - была совершена покупка
* `send` - письмо было отправлено
* `unsubscribe` - пользователь отписался от рассылки
* `hbq_spam` - письмо попало в спам
* `hard_bounce` - серьезная ошибка в доставке сообщения, письмо не будет доставлено
* `subscribe` - подписка на рассылку
* `soft_bounce` - временная ошибка доставки письма, будет повторная попытка отправки
* `complain` - поступила жалоба
* `close` - сообщение закрыто? (тут не понятно)

Можем посчитать, например, конверсию - число покупок от общего числа рассылок или от общего числа открытых рассылок. Разделим действия с письмом на `позитивные` - **перешел по ссылке**, **подписался** или **совершил покупку**, `нейтральные` - **отправлено** или **открыто** и `негативные` - **все остальное**. Соответственно можем сформировать признаки: **доля позитивных/нейтральных/негативных действий**, **доля покупок от всех рассылок/от позитивных действий**, **доля (переходов по ссылке + покупок) от всех рассылок**.

Варианты признаков для использования моделью

* `total_messages` - общее число рассылок, отправленных клиенту
* `positive_share` - доля позитивных действий от общего число рассылок
* `neutral_share` - доля нейтральных действий от общего числа рассылок
* `purchase_positive_share` - доля покупок от числа позитивных действий
* `purchase_share` - доля покупок от общего числа рассылок
* `click_share` - доля переходов по ссылке/покупок от общего числа рассылок
* `common_channel` - наиболее частый путь получения рассылок для данного клиента. `email`, `mobile_push` или `mixed`, если доля одного из них в пределах (0.25, 0.75).

#### 3. `target`

Размер и первые строки

In [120]:
target_table = dataframes[2]
print(target_table.shape)
display(target_table.head())

(49849, 2)


,client_id,target
0,1515915625468060902,0
1,1515915625468061003,1
2,1515915625468061099,0
3,1515915625468061100,0
4,1515915625468061170,0


* `client_id` - идентификатор клиента
* `target` - клиент совершил покупку в целевом периоде

#### 4. `full_campaign_daily_event`

Размер и первые строки

In [121]:
full_campaign_daily_event = dataframes[3]
print(full_campaign_daily_event.shape)
display(full_campaign_daily_event.sample(5).T)

(131072, 24)


,87989,83078,20654,127233,97987
date,2023-10-03,2023-08-13,2022-09-10,2024-04-29,2023-12-24
bulk_campaign_id,13580,13640,5286,15007,13471
count_click,6,315,5,12,0
count_complain,0,0,0,0,0
count_hard_bounce,0,4,0,0,0
count_open,140,8195,20,0,43
count_purchase,0,4,0,0,0
count_send,0,0,0,0,0
count_soft_bounce,0,13,0,0,0
count_subscribe,0,14,0,0,0


* `date` - дата
* `bulk_campaign_id` - идентификатор рассылки
* `count_event*` - общее количество каждого события event
* `nunique_event*` - количество уникальных client_id в каждом событии

#### 5. `full_campaign_daily_event_channel`

Размер и первые строки таблицы

In [122]:
full_campaign_daily_event_channel = dataframes[4]
print(full_campaign_daily_event_channel.shape)
display(full_campaign_daily_event_channel.sample(5).T)

(131072, 36)


,38763,64343,76530,34490,48692
date,2022-12-01,2023-03-25,2023-06-01,2022-11-13,2023-01-11
bulk_campaign_id,2716,9540,4591,4492,4696
count_click_email,0,1,0,0,0
count_click_mobile_push,0,0,0,0,0
count_open_email,5,138,1,15,7
count_open_mobile_push,0,0,0,0,0
count_purchase_email,0,0,0,0,0
count_purchase_mobile_push,0,0,0,0,0
count_soft_bounce_email,0,0,0,0,0
count_subscribe_email,0,0,0,0,0


* `date` - дата
* `bulk_campaign_id` - идентификатор рассылки
* `count_event*_channel*` - общее количество каждого события по каналам
* `nunique_event*_channel*` - количество уникальных client_id по событиям и каналам

Для клиентов планируется сформировать поле `common_channel`. Если `common_channel = email` или `common_channel = mobile_push`, то используем данные из таблицы 5, если `common_channel = mixed`, то используем данные из таблицы 4. 

Идея состоит в том, чтобы расчитывать то, насколько действие клиента типично для рассылки в целом. Например, если по ссылке в рассылке перешли 90% клиентов, то клиент, который не перешел, будет иметь заведомо ниже вероятность совершить покупку, чем остальные. Например, по ссылке перешли `X` (доля) всех клиентов. Тогда если клиент совершил `нейтральное действие`, то он получит 0 баллов. Если он совершил `позитивное действие`, то его баллы будут равны стоимости этого `позитивного действия` (заранее отпределенная), деленной на `X` (т. е. чем меньше клиентов перешло по ссылке, тем каждый из этих клиентов более вероятно совершит покупку в будущем). Если он совершил `негативное действие`,  то его баллы будут равны минус стоимости этого `негативного действия`, деленной на `(1-X)`. 

Суммируя баллы по всем рассылкам мы получим значение новой метрики для каждого отдельного клиента. Изменяя стоимости отдельных действий можем изменять и саму метрику

Варианты признаков для использования моделью

* `weighted_effect` - значение суммарной метрики метрики, рассчитанной по описанному выше методу. 
* `weighted_positive_effect` - суммарный вклад позитивных действий
* `weighted_negative_effect` - суммарный вклда негативных действий

`weights` веса для всех действий клиента, например:

| Event | Value |
|-------|-------|
| `open` | 0.1 |
| `click` | 1 |
| `purchase` | 10 |
| `send` | 0 |
| `unsubscribe` | -10 |
| `hbq_spam` | -10 |
| `hard_bounce` | -20 |
| `subscribe` | 5 |
| `soft_bounce` | -5 |
| `complain` | -20 |
| `close` | -5 |

### Предобработка

#### `apparent_purchases`

Пропуски, Дубликаты и Типы данных

In [123]:
display(apparel_purchases.isna().sum())
display(apparel_purchases.duplicated().sum())
values_and_type(apparel_purchases)

client_id       0
quantity        0
price           0
category_ids    0
date            0
message_id      0
dtype: int64

73020

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,client_id,int64,1515915625853312319,1515915625624308268,1515915625607113301,1515915625500232103,1515915625470860261
1,quantity,int64,1,2,3,4,6
2,price,float64,999.0,1499.0,1999.0,699.0,599.0
3,category_ids,object,"['4', '28', '57', '431']","['4', '28', '260', '420']","['4', '28', '244', '432']",[],"['4', '28', '275', '421']"
4,date,object,2022-11-11,2023-06-10,2023-04-28,2022-11-15,2022-12-22
5,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625624308268-7803-636dfea7ca890,1515915625880116527-8725-639b2f0ad5e2e,1515915625607113301-13818-650a8af70fd67,1515915625558691508-14212-6567128370bdd


Добавим новое поле с итоговой ценой заказа

In [124]:
apparel_purchases['sum_price'] = apparel_purchases['quantity'] * apparel_purchases['price']

Проверка

In [125]:
values_and_type(apparel_purchases)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,client_id,int64,1515915625853312319,1515915625624308268,1515915625607113301,1515915625500232103,1515915625470860261
1,quantity,int64,1,2,3,4,6
2,price,float64,999.0,1499.0,1999.0,699.0,599.0
3,category_ids,object,"['4', '28', '57', '431']","['4', '28', '260', '420']","['4', '28', '244', '432']",[],"['4', '28', '275', '421']"
4,date,object,2022-11-11,2023-06-10,2023-04-28,2022-11-15,2022-12-22
5,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625624308268-7803-636dfea7ca890,1515915625880116527-8725-639b2f0ad5e2e,1515915625607113301-13818-650a8af70fd67,1515915625558691508-14212-6567128370bdd
6,sum_price,float64,999.0,1499.0,1999.0,699.0,599.0


Пропусков нет, дупликаты - это не ошибка, а много одинаковых заказов в один день. Дату нужно привести к типу даты

In [126]:
apparel_purchases['date'] = pd.to_datetime(apparel_purchases['date'], format='%Y-%m-%d')

Проверка

In [127]:
values_and_type(apparel_purchases)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,client_id,int64,1515915625853312319,1515915625624308268,1515915625607113301,1515915625500232103,1515915625470860261
1,quantity,int64,1,2,3,4,6
2,price,float64,999.0,1499.0,1999.0,699.0,599.0
3,category_ids,object,"['4', '28', '57', '431']","['4', '28', '260', '420']","['4', '28', '244', '432']",[],"['4', '28', '275', '421']"
4,date,datetime64[ns],2022-11-11 00:00:00,2023-06-10 00:00:00,2023-04-28 00:00:00,2022-11-15 00:00:00,2022-12-22 00:00:00
5,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625624308268-7803-636dfea7ca890,1515915625880116527-8725-639b2f0ad5e2e,1515915625607113301-13818-650a8af70fd67,1515915625558691508-14212-6567128370bdd
6,sum_price,float64,999.0,1499.0,1999.0,699.0,599.0


`category_ids` имеет тип `object` и является строкой. Преобразуем в `list`

In [128]:
def str_to_list(stroka):
    if len(stroka) > 6:
        split = stroka.split("'")
        #print(split)
        if len(split) < 4:
            numbers = [int(split[1])]
        elif len(split) < 6:
            numbers = [int(split[1]), int(split[3])]
        elif len(split) < 8:
            numbers = [int(split[1]), int(split[3]), int(split[5])]
        else:
            numbers = [int(split[1]), int(split[3]), int(split[5]), int(split[7])]
            
        #print(numbers)
        return numbers
    else:
        return []

In [129]:
n_jobs = -1  # Использовать все доступные ядра
apparel_purchases['category_ids'] = Parallel(n_jobs=n_jobs)(
    delayed(str_to_list)(x) for x in apparel_purchases['category_ids']
)

Проверка

In [130]:
values_and_type(apparel_purchases)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,client_id,int64,1515915625853312319,1515915625624308268,1515915625607113301,1515915625500232103,1515915625470860261
1,quantity,int64,1,2,3,4,6
2,price,float64,999.0,1499.0,1999.0,699.0,599.0
3,category_ids,object,"[4, 28, 57, 431]","[4, 28, 260, 420]",[],"[4, 28, 244, 432]","[4, 28, 275, 421]"
4,date,datetime64[ns],2022-11-11 00:00:00,2023-06-10 00:00:00,2023-04-28 00:00:00,2022-11-15 00:00:00,2022-12-22 00:00:00
5,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625624308268-7803-636dfea7ca890,1515915625880116527-8725-639b2f0ad5e2e,1515915625607113301-13818-650a8af70fd67,1515915625558691508-14212-6567128370bdd
6,sum_price,float64,999.0,1499.0,1999.0,699.0,599.0


#### `apparel_messages`

Пропуски, Дубликаты и Типы данных

In [131]:
display(apparel_messages.isna().sum())
display(apparel_messages.duplicated().sum())
values_and_type(apparel_messages)

bulk_campaign_id    0
client_id           0
message_id          0
event               0
channel             0
date                0
created_at          0
dtype: int64

48610

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,bulk_campaign_id,int64,14272,14276,4679,11760,14081
1,client_id,int64,1515915625516327994,1515915625625548006,1515915625804998560,1515915625489071904,1515915625489095763
2,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625490540122-6973-633edf707840d,1515915625490241385-6973-633edf70723d8,1515915625629509124-6374-63241bd1541d5,1515915625488270582-3433-6232d6007e929
3,event,object,send,open,click,purchase,hard_bounce
4,channel,object,mobile_push,email,-,-,-
5,date,object,2023-06-10,2024-01-26,2023-12-11,2023-12-10,2023-12-26
6,created_at,object,2023-12-29 15:20:53,2023-07-03 10:22:53,2023-12-29 14:51:53,2023-12-29 14:51:33,2023-12-29 15:20:13


Также нет пропусков, дупликаты имеют смысл того, что из одной рассылки было отправлено несколько сообщений. Дату необходимо привести к типу даты

In [132]:
apparel_messages['date'] = pd.to_datetime(apparel_messages['date'], format='%Y-%m-%d')

Проверка

In [133]:
values_and_type(apparel_messages)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,bulk_campaign_id,int64,14272,14276,4679,11760,14081
1,client_id,int64,1515915625516327994,1515915625625548006,1515915625804998560,1515915625489071904,1515915625489095763
2,message_id,object,1515915625489095763-6251-6311b13a4cf78,1515915625490540122-6973-633edf707840d,1515915625490241385-6973-633edf70723d8,1515915625629509124-6374-63241bd1541d5,1515915625488270582-3433-6232d6007e929
3,event,object,send,open,click,purchase,hard_bounce
4,channel,object,mobile_push,email,-,-,-
5,date,datetime64[ns],2023-06-10 00:00:00,2024-01-26 00:00:00,2023-12-11 00:00:00,2023-12-10 00:00:00,2023-12-26 00:00:00
6,created_at,object,2023-12-29 15:20:53,2023-07-03 10:22:53,2023-12-29 14:51:53,2023-12-29 14:51:33,2023-12-29 15:20:13


#### `full_campaign_daily_event`

Пропуски, Дубликаты и Типы данных

In [134]:
display(full_campaign_daily_event.isna().sum())
display(full_campaign_daily_event.duplicated().sum())
values_and_type(full_campaign_daily_event)

date                   0
bulk_campaign_id       0
count_click            0
count_complain         0
count_hard_bounce      0
count_open             0
count_purchase         0
count_send             0
count_soft_bounce      0
count_subscribe        0
count_unsubscribe      0
nunique_click          0
nunique_complain       0
nunique_hard_bounce    0
nunique_open           0
nunique_purchase       0
nunique_send           0
nunique_soft_bounce    0
nunique_subscribe      0
nunique_unsubscribe    0
count_hbq_spam         0
nunique_hbq_spam       0
count_close            0
nunique_close          0
dtype: int64

0

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,date,object,2023-01-24,2023-02-12,2023-02-09,2023-01-27,2023-01-10
1,bulk_campaign_id,int64,2499,3557,5286,3401,4679
2,count_click,int64,0,1,2,3,4
3,count_complain,int64,0,1,2,3,4
4,count_hard_bounce,int64,0,1,2,3,4
5,count_open,int64,0,1,2,3,4
6,count_purchase,int64,0,1,2,3,4
7,count_send,int64,0,1,11520,4,2
8,count_soft_bounce,int64,0,1,2,3,4
9,count_subscribe,int64,0,1,2,3,4


Нет пропусков, дубликатов. Дату приведем к типу даты

In [135]:
full_campaign_daily_event['date'] = pd.to_datetime(full_campaign_daily_event['date'], format='%Y-%m-%d')

Проверка

In [136]:
values_and_type(full_campaign_daily_event)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,date,datetime64[ns],2023-01-24 00:00:00,2023-02-12 00:00:00,2023-02-09 00:00:00,2023-01-27 00:00:00,2023-01-10 00:00:00
1,bulk_campaign_id,int64,2499,3557,5286,3401,4679
2,count_click,int64,0,1,2,3,4
3,count_complain,int64,0,1,2,3,4
4,count_hard_bounce,int64,0,1,2,3,4
5,count_open,int64,0,1,2,3,4
6,count_purchase,int64,0,1,2,3,4
7,count_send,int64,0,1,11520,4,2
8,count_soft_bounce,int64,0,1,2,3,4
9,count_subscribe,int64,0,1,2,3,4


#### `full_campaign_daily_event_channel`

Пропуски, Дубликаты и Типы данных

In [137]:
display(full_campaign_daily_event_channel.isna().sum())
display(full_campaign_daily_event_channel.duplicated().sum())
values_and_type(full_campaign_daily_event_channel)

date                               0
bulk_campaign_id                   0
count_click_email                  0
count_click_mobile_push            0
count_open_email                   0
count_open_mobile_push             0
count_purchase_email               0
count_purchase_mobile_push         0
count_soft_bounce_email            0
count_subscribe_email              0
count_unsubscribe_email            0
nunique_click_email                0
nunique_click_mobile_push          0
nunique_open_email                 0
nunique_open_mobile_push           0
nunique_purchase_email             0
nunique_purchase_mobile_push       0
nunique_soft_bounce_email          0
nunique_subscribe_email            0
nunique_unsubscribe_email          0
count_hard_bounce_mobile_push      0
count_send_mobile_push             0
nunique_hard_bounce_mobile_push    0
nunique_send_mobile_push           0
count_hard_bounce_email            0
count_hbq_spam_email               0
count_send_email                   0
n

0

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,date,object,2023-01-24,2023-02-12,2023-02-09,2023-01-27,2023-01-10
1,bulk_campaign_id,int64,2499,3557,5286,3401,4679
2,count_click_email,int64,0,1,2,3,4
3,count_click_mobile_push,int64,0,1,2,3,4
4,count_open_email,int64,0,1,2,3,4
5,count_open_mobile_push,int64,0,1,2,3,4
6,count_purchase_email,int64,0,1,2,3,4
7,count_purchase_mobile_push,int64,0,1,2,3,4
8,count_soft_bounce_email,int64,0,1,2,3,4
9,count_subscribe_email,int64,0,1,2,3,4


Нет пропусков, дубликатов, только дату нужно привести к типу даты

In [138]:
full_campaign_daily_event_channel['date'] = pd.to_datetime(full_campaign_daily_event_channel['date'], format='%Y-%m-%d')

Проверка

In [139]:
values_and_type(full_campaign_daily_event_channel)

,column_name,type,value_1,value_2,value_3,value_4,value_5
0,date,datetime64[ns],2023-01-24 00:00:00,2023-02-12 00:00:00,2023-02-09 00:00:00,2023-01-27 00:00:00,2023-01-10 00:00:00
1,bulk_campaign_id,int64,2499,3557,5286,3401,4679
2,count_click_email,int64,0,1,2,3,4
3,count_click_mobile_push,int64,0,1,2,3,4
4,count_open_email,int64,0,1,2,3,4
5,count_open_mobile_push,int64,0,1,2,3,4
6,count_purchase_email,int64,0,1,2,3,4
7,count_purchase_mobile_push,int64,0,1,2,3,4
8,count_soft_bounce_email,int64,0,1,2,3,4
9,count_subscribe_email,int64,0,1,2,3,4


Создание нового датафрейма

#### `Apparel_purchases`

In [140]:
new_columns_apparel_purchases = target_table.copy()

Создаем новые признаки из таблицы `apparel_purchases`:
* `total_quantity_period*` - общее число покупок за период. Можно рассмотреть следующие периоды - неделя, месяц, 3 месяца, полгода.
* `total_spent_period*` - общее количество потраченных средств за период. `period*` - промежуток времени, неделя, месяц, 3 месяца, полгода.
* `avg_quantity_period*` - среднее число товаров за период (можем суммарное число товаров делить на длительность периода или на число только тех дней, когда были заказы). `period*` - промежуток времени, неделя, месяц, 3 месяца, полгода.
* `avg_quantity_per_order` - среднее число товаров за заказ
* `avg_spent_period*` -  среднее число потраченных средств за период (можем сумму потраченных средств делить на длительность периода или на число только тех дней, когда были заказы). `period*` - промежуток времени, неделя, месяц, 3 месяца, полгода.
* `best_categories_level_number*` - наиболее популярная категория каждого уровня у человека. Для каждого уровня берутся все категории, которые встречаются на этом уровне, затем считается их суммарная встречаемость на всех уровнях. Выбирается самая встречаемая категория по сумме на всех уровнях. `number*` - номер уровня от 1 до 4.
* `days_after_last_purchase` - число дней с последней покупки.


Максимальная дата, используем как текущую

In [141]:
max_date = apparel_purchases.date.max()
print(max_date) 

2024-02-16 00:00:00


Функция для создания новых признаков

In [142]:
def get_periods_data(apparel_purchases, max_date, period_name, period, agg_column, name_prefix='total_quantity_', aggfunc='sum'):
    period_length = period
    min_date = max_date - timedelta(days=period_length)
    df = (apparel_purchases[apparel_purchases['date'] >= min_date]
                        .groupby(by='client_id')[agg_column]
                        .agg(aggfunc)
                        .reset_index(name=name_prefix + period_name))
    return df

Списки периодов

In [143]:
period_names = ['week', 'month', '3_month', 'half_year', 'year', '3_years']
periods = [7, 30, 90, 180, 360, 1080]

`total_quantity`

In [144]:
total_quantity_columns = target_table.copy().drop(columns=['target'])
for i in range(len(period_names)):
    total_quantity_columns = pd.merge(total_quantity_columns, get_periods_data(apparel_purchases, max_date, period_names[i], periods[i], 'quantity'), on='client_id', how='left')
total_quantity_columns.fillna(0, inplace=True)

In [145]:
display(total_quantity_columns.sample(5))

,client_id,total_quantity_week,total_quantity_month,total_quantity_3_month,total_quantity_half_year,total_quantity_year,total_quantity_3_years
19075,1515915625500537241,0.0,0.0,0.0,0.0,0.0,1
48922,1515915625990517314,0.0,2.0,2.0,2.0,2.0,2
732,1515915625468118696,0.0,0.0,0.0,0.0,0.0,2
49687,1515915626007074111,0.0,0.0,1.0,1.0,1.0,1
37968,1515915625642188604,0.0,0.0,0.0,0.0,0.0,1


`total_spent_period`

In [146]:
total_spent_columns = target_table.copy().drop(columns=['target'])
for i in range(len(period_names)):
    total_spent_columns = pd.merge(total_spent_columns, \
        get_periods_data(apparel_purchases, max_date, period_names[i], periods[i], 'sum_price', name_prefix='total_spent_'),\
            on='client_id', how='left')
total_spent_columns.fillna(0, inplace=True)

In [147]:
display(total_spent_columns.sample(5))

,client_id,total_spent_week,total_spent_month,total_spent_3_month,total_spent_half_year,total_spent_year,total_spent_3_years
43228,1515915625807176103,0.0,0.0,0.0,0.0,0.0,997.0
43741,1515915625815847454,0.0,0.0,0.0,0.0,6996.0,6996.0
47883,1515915625974413898,0.0,0.0,0.0,0.0,1699.0,1699.0
20815,1515915625503037462,0.0,0.0,0.0,0.0,0.0,1398.0
39691,1515915625703934028,0.0,0.0,0.0,0.0,15900.0,15900.0


`avg_quantity_period`

In [148]:
avg_quantity_columns = target_table.copy().drop(columns=['target'])
for i in range(len(period_names)):
    avg_quantity_columns = pd.merge(avg_quantity_columns, \
        get_periods_data(apparel_purchases, max_date, period_names[i], periods[i], 'quantity', name_prefix='avg_quantity_', aggfunc=lambda x: x.mean()),\
            on='client_id', how='left')
avg_quantity_columns.fillna(0, inplace=True)

In [149]:
display(avg_quantity_columns.sample(5))

,client_id,avg_quantity_week,avg_quantity_month,avg_quantity_3_month,avg_quantity_half_year,avg_quantity_year,avg_quantity_3_years
35738,1515915625604418355,0.0,0.0,0.0,0.0,0.0,1.0
21643,1515915625509018725,0.0,0.0,0.0,0.0,0.0,1.0
4245,1515915625474727491,0.0,0.0,0.0,0.0,1.0,1.0
3873,1515915625473620812,0.0,0.0,0.0,0.0,0.0,1.0
32489,1515915625582801333,0.0,0.0,0.0,0.0,1.0,1.0


`avg_quantity_per_order`

In [150]:
avg_quantity_per_order_columns = target_table.copy().drop(columns=['target'])
avg_quantity_per_order_columns = pd.merge(avg_quantity_per_order_columns, (apparel_purchases
                                                                 .groupby(by='client_id')['quantity']
                                                                 .mean()
                                                                 .reset_index(name='avg_quantity_per_order')), 
                                on='client_id', how='left')

In [151]:
display(avg_quantity_per_order_columns.sort_values(by='avg_quantity_per_order', ascending=False).head(5))

,client_id,avg_quantity_per_order
21693,1515915625509511395,19.000000
41712,1515915625776529170,18.000000
37043,1515915625629378288,15.000000
12512,1515915625489995179,10.000000
40385,1515915625740853027,7.333333


`avg_spent_period`

In [152]:
avg_spent_columns = target_table.copy().drop(columns=['target'])
for i in range(len(period_names)):
    avg_spent_columns = pd.merge(avg_spent_columns, \
        get_periods_data(apparel_purchases, max_date, period_names[i], periods[i], 'sum_price', name_prefix='avg_spent_', aggfunc=lambda x: x.mean()),\
            on='client_id', how='left')
avg_spent_columns.fillna(0, inplace=True)

In [153]:
display(avg_spent_columns.sample(5))

,client_id,avg_spent_week,avg_spent_month,avg_spent_3_month,avg_spent_half_year,avg_spent_year,avg_spent_3_years
36459,1515915625623207816,0.0,0.0,0.0,0.0,0.000000,3099.000000
22450,1515915625536060557,0.0,0.0,0.0,0.0,0.000000,1999.000000
43262,1515915625807771990,0.0,0.0,0.0,0.0,0.000000,1819.000000
19059,1515915625500527259,0.0,0.0,0.0,0.0,600.666667,600.666667
40848,1515915625764063941,0.0,0.0,0.0,0.0,0.000000,1499.000000


`best_categories_level_number`

Для описания способа обработки случая, когда одна и та же категория встречается на разных уровнях, рассмотрим следующий пример для категорий покупок одного клиента:

| категория 1 | категория 2 | категория 3 | категория 4 |
|-------------------|-------------------|-------------------|-------------------|
| 1                 | 7                 | `20`                | 60                |
| 1                 | 7                 | `20`                | 61                |
| 1                 | 7                 | `20`                | 62                |
| 1                 | `20`                | 44                  | 88                |
| 1                 | `20`                | 45                  | 110               |
| 1                 | 4                 | 12                  | `20`                |

Категория `20` встречается итого 6 раз: 3 раза на 3-м уровне, 2 раза на 2-м уровне и 1 раз на 4-м уровне. Замечу, что зачастую количество раз, которое категория встречается на 1-м уровне, больше, чем число раз, которое категория встречается на 2-м уровне. Хотя бы потому, что на уровне 2 больше категорий, чем на 1-м уровне, а число заказов одно и то же. То есть если мы прибавим к числу заказов на 2-м уровне число заказов на 1-м уровне, то довольно вероятно, что эта категория окажется наиболее популярной на уровне 2. Так что имеет смысл только прибавлять только к категории с меньшим уровнем категорию с большим уровнем. 

В данном примере:

`(3 + 2 + 1) = 6` - считаем, что именно столько раз категория `20` встречалась на 2-м уровне. (сумма по всем уровням от 2 до 4)

`(2 + 1) = 3` - считаем, что именно столько раз категория `20` встречалась на 3-м уровне. (сумма числа раз, которое категория встречается на 3-м и 4-м уровнях)

`1` - именно столько раз категория встретилась на 4-м уровне.




Новое поле будем создавать вручную, а не с помощью функций pandas

In [154]:
apparel_purchases.sample(5)

,client_id,quantity,price,category_ids,date,message_id,sum_price
59368,1515915625867955942,1,899.0,"[4, 29, 309, 939]",2022-11-17,1515915625491254994-7879-6374873f830ae,899.0
186757,1515915625558004095,1,699.0,"[4, 28, 213, 436]",2023-12-24,1515915625558004095-14359-6587f2cc44bdb,699.0
95317,1515915625590690250,1,700.0,"[4, 28, 260, 420]",2023-03-24,1515915625488647496-11209-641d582511d3e,700.0
128769,1515915625516627799,1,4499.0,"[4, 28, 57, 431]",2023-06-06,1515915625516627799-12864-64708ae259ae3,4499.0
91396,1515915625865927579,1,827.0,"[4, 28, 275, 673]",2023-03-13,1515915625865927579-10792-640ad975a381d,827.0


In [155]:
def get_category_value(x, j):
    result = x[j] if len(x) > j else None
    #print(f"x: {x}, j: {j}, result: {result}")  # отладочный вывод
    return result

In [156]:
def process_client_data(client_id, group_df):
    """Обработка данных одного клиента (работает с уже отфильтрованной группой)"""
    unique_categories = []
    curr_data = group_df[['client_id', 'category_ids']].copy()
    
    if curr_data.empty:
        return client_id, {}, {}, {}
    
    n_cat = curr_data['category_ids'].apply(lambda x: len(x)).max()
    if pd.isna(n_cat) or n_cat == 0:
        return client_id, {}, {}, {}
    
    n_cat = int(n_cat)
    curr_data['category_ids'] = curr_data['category_ids'].apply(lambda x: x + [None]*(n_cat-len(x)) if len(x) < n_cat else x)
    
    for j in range(n_cat):
        curr_data['category_' + str(j)] = curr_data['category_ids'].apply(lambda x: get_category_value(x, j))
    
    for cats in curr_data['category_ids'].to_list():
        unique_categories.extend(cats)
    unique_categories = list(set(unique_categories))
    
    if not unique_categories:
        return client_id, {}, {}, {}
    
    categories_count = np.zeros((len(unique_categories), n_cat))
    
    # Расчет того, сколько раз каждая категория встречается на каждом уровне
    for i in range(len(unique_categories)):
        curr_category = unique_categories[i]
        for j in range(n_cat):
            categories_count[i, j] = curr_data.loc[curr_data['category_' + str(j)] == curr_category, 'category_' + str(j)].count()
    
    # Обновление массива по алгоритму
    for i in range(len(unique_categories)):
        for j in range(n_cat):
            if categories_count[i, j] != 0:
                categories_count[i, j] = np.sum(categories_count[i, j:n_cat])  
    
    # Нахождение лучших категорий для каждого уровня
    client_values = {}
    client_categories = {}
    client_percentages = {}
    for i in range(n_cat):
        client_values[i] = np.max(categories_count[:, i])
        client_categories[i] = unique_categories[np.argmax(categories_count[:, i])]
        total_sum = np.sum(categories_count[:, i])
        client_percentages[i] = (client_values[i] / total_sum * 100) if total_sum > 0 else 0
    
    return client_id, client_values, client_categories, client_percentages

# Шаг 1: Подготавливаем данные один раз (избегаем повторной фильтрации)
print("Подготовка данных по клиентам...")
grouped_data = list(apparel_purchases.groupby('client_id', sort=False))
print(f"Разбито на {len(grouped_data)} групп")

# Шаг 2: Параллельная обработка с joblib (более эффективно работает с pandas/numpy)
print(f"Запуск параллельной обработки...")
start_time = time.time()

results = Parallel(n_jobs=-1, verbose=1, batch_size='auto')(
    delayed(process_client_data)(client_id, group) 
    for client_id, group in grouped_data
)

elapsed_time = time.time() - start_time
print(f"\nОбработка завершена за {elapsed_time:.2f} сек!")

# Шаг 3: Собираем результаты
values = {}
categories = {}
percentages = {}
for client_id, client_values, client_categories, client_percentages in results:
    values[client_id] = [client_values.get(i, 0) for i in range(4)]
    categories[client_id] = [client_categories.get(i, 0) for i in range(4)]
    percentages[client_id] = [client_percentages.get(i, 0) for i in range(4)]

print(f"Успешно обработано {len(values)} клиентов")

Подготовка данных по клиентам...
Разбито на 49849 групп
Запуск параллельной обработки...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 260 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 6100 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done 17300 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done 31700 tasks      | elapsed:   15.7s
[Parallel(n_jobs=-1)]: Done 48850 tasks      | elapsed:   23.1s



Обработка завершена за 23.70 сек!
Успешно обработано 49849 клиентов


[Parallel(n_jobs=-1)]: Done 49810 out of 49849 | elapsed:   23.6s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 49849 out of 49849 | elapsed:   23.6s finished


In [157]:
unique_client_ids = target_table['client_id'].unique()
best_categories_level_columns = pd.DataFrame(data=unique_client_ids, columns=['client_id'])

# Создаем все колонки
for i in range(4):
    best_categories_level_columns['best_category_level_' + str(i+1)] = best_categories_level_columns['client_id'].apply(lambda x: categories[x][i])
    best_categories_level_columns['best_count_level_' + str(i+1)] = best_categories_level_columns['client_id'].apply(lambda x: values[x][i])
    best_categories_level_columns['best_percentage_level_' + str(i+1)] = best_categories_level_columns['client_id'].apply(lambda x: percentages[x][i])

# Переупорядочиваем колонки: категория, количество и процент чередуются для каждого уровня
columns_order = ['client_id']
for i in range(4):
    columns_order.append(f'best_category_level_{i+1}')
    columns_order.append(f'best_count_level_{i+1}')
    columns_order.append(f'best_percentage_level_{i+1}')

best_categories_level_columns = best_categories_level_columns[columns_order]

print(f"Размеры датафрейма: {best_categories_level_columns.shape}")
print(f"Колонки: {best_categories_level_columns.columns.tolist()}")
display(best_categories_level_columns.sample(10))

Размеры датафрейма: (49849, 13)
Колонки: ['client_id', 'best_category_level_1', 'best_count_level_1', 'best_percentage_level_1', 'best_category_level_2', 'best_count_level_2', 'best_percentage_level_2', 'best_category_level_3', 'best_count_level_3', 'best_percentage_level_3', 'best_category_level_4', 'best_count_level_4', 'best_percentage_level_4']


,client_id,best_category_level_1,best_count_level_1,best_percentage_level_1,best_category_level_2,best_count_level_2,best_percentage_level_2,best_category_level_3,best_count_level_3,best_percentage_level_3,best_category_level_4,best_count_level_4,best_percentage_level_4
44484,1515915625841919208,5562,2.0,100.0,5558,2.0,100.000000,5582,2.0,100.000000,1111,2.0,100.000000
29665,1515915625567291798,2,3.0,100.0,17,2.0,66.666667,232,1.0,33.333333,525,1.0,33.333333
27698,1515915625559764791,2,2.0,100.0,17,2.0,100.000000,180,2.0,100.000000,465,2.0,100.000000
33490,1515915625587151192,4,2.0,100.0,29,2.0,100.000000,310,2.0,100.000000,500,2.0,100.000000
17680,1515915625491899218,4,1.0,100.0,28,1.0,100.000000,146,1.0,100.000000,548,1.0,100.000000
28843,1515915625565391647,5562,4.0,100.0,5632,3.0,75.000000,5552,2.0,50.000000,1298,2.0,50.000000
34504,1515915625590358059,4,2.0,100.0,28,2.0,100.000000,239,2.0,100.000000,418,2.0,100.000000
47684,1515915625970792937,5562,2.0,100.0,5589,2.0,100.000000,5586,2.0,100.000000,1767,2.0,100.000000
12348,1515915625489932715,5562,1.0,100.0,5683,1.0,100.000000,5556,1.0,100.000000,1052,1.0,100.000000
26967,1515915625559301566,4,2.0,100.0,28,2.0,100.000000,275,2.0,100.000000,673,2.0,100.000000


`days_after_last_purchase`

In [158]:
days_after_last_purchase_columns = apparel_purchases.groupby('client_id')['date'].max().reset_index(name='last_purchase_date')
days_after_last_purchase_columns['days_after_last_purchase'] = (max_date - days_after_last_purchase_columns['last_purchase_date']).dt.days
days_after_last_purchase_columns.drop(columns=['last_purchase_date'], inplace=True)
display(days_after_last_purchase_columns.sample(5))

,client_id,days_after_last_purchase
31728,1515915625579545611,44
11584,1515915625489649567,542
20555,1515915625502573631,232
24401,1515915625553009322,360
42583,1515915625794070352,431


#### `Apparel_messages`

Новые признаки:
* `total_messsages` -  общее число сообщений, отправленных клиенту
* `positive_share` - доля позитивных действий от общего число рассылок
* `neutral_share` - доля нейтральных действий от общего числа рассылок
* `purchase_positive_share` - доля покупок от числа позитивных действий
* `purchase_share` - доля покупок от общего числа рассылок
* `click_share` - доля переходов по ссылке/покупок от общего числа рассылок
* `common_channel` - наиболее частый путь получения рассылок для данного клиента. `email`, `mobile_push` или `mixed`, если доля одного из них в пределах (0.25, 0.75).

`total_messages`

In [159]:
total_messages_columns = apparel_messages.groupby('client_id')['event'].count().reset_index(name='total_messages')
display(total_messages_columns.sample(5))

,client_id,total_messages
7697,1515915625487744680,7
12230,1515915625489265193,183
20233,1515915625491596445,307
31494,1515915625560287197,355
28958,1515915625558400332,85


`positive_share`

Значения поля `event`
* `open` - письмо/уведомление было открыто
* `click` - пользователь перешел по ссылке в письме/уведомлении
* `purchase` - была совершена покупка
* `send` - письмо было отправлено
* `unsubscribe` - пользователь отписался от рассылки
* `hbq_spam` - письмо попало в спам
* `hard_bounce` - серьезная ошибка в доставке сообщения, письмо не будет доставлено
* `subscribe` - подписка на рассылку
* `soft_bounce` - временная ошибка доставки письма, будет повторная попытка отправки
* `complain` - поступила жалоба
* `close` - сообщение закрыто? (тут не понятно)

Задаем списки позитивных, нейтральных и негативных действий

In [160]:
actions = ['open', 'click', 'purchase', 'send', 'unsubscribe', 'hbq_spam', 'hard_bounce', 'subscribe', 'soft_bounce', 'complain', 'close']
positive = ['open', 'click', 'purchase', 'subscribe']
negative = ['unsubscribe', 'hbq_spam', 'hard_bounce', 'soft_bounce', 'complain', 'close']
neutral = ['send']

Создаем новое поле

In [161]:
apparel_messages['action_type3'] = apparel_messages['event'].apply(lambda x: 'positive' if x in positive else ('negative' if x in negative else 'neutral'))
apparel_messages['action_type2'] = apparel_messages['event'].apply(lambda x: 'positive' if x in positive + neutral else 'negative')
display(apparel_messages.sample(5))

,bulk_campaign_id,client_id,message_id,event,channel,date,created_at,action_type3,action_type2
4449701,11132,1515915625487841894,1515915625487841894-11132-6419d914c4930,open,mobile_push,2023-03-21,2023-03-21 16:23:22,positive,positive
6879233,13610,1515915625570231633,1515915625570231633-13610-64d4df7f332d6,send,mobile_push,2023-08-10,2023-08-10 14:01:53,neutral,positive
5124115,12685,1515915625487223296,1515915625487223296-12685-64663e91dc0ab,open,mobile_push,2023-05-18,2023-05-18 15:08:36,positive,positive
4909217,12184,1515915625490054932,1515915625490054932-12184-644a2fccd8e67,send,email,2023-04-27,2023-04-27 08:20:03,neutral,positive
11335554,14446,1515915625468226285,1515915625468226285-14446-659cf9f9ede13,send,mobile_push,2024-01-09,2024-01-09 07:49:13,neutral,positive


Формируем таблицу с новыми признаками

In [ ]:
print("Расчет positive_share...")
start_time = time.time()

positive_share_columns_2 = apparel_messages.groupby('client_id')['action_type2'].apply(lambda x: (x == 'positive').sum() / len(x) * 100).reset_index(name='positive_share_2')
positive_share_columns_3 = apparel_messages.groupby('client_id')['action_type3'].apply(lambda x: (x == 'positive').sum() / len(x) * 100).reset_index(name='positive_share_3')
positive_share_columns = pd.merge(positive_share_columns_2, positive_share_columns_3, on='client_id', how='left')

elapsed_time = time.time() - start_time
print(f"Расчет завершен за {elapsed_time:.2f} сек!")
display(positive_share_columns.sample(5))

Расчет positive_share...
Расчет завершен за 7.52 сек!


,client_id,positive_share_2,positive_share_3
20135,1515915625491559317,100.000000,22.525597
10415,1515915625488741920,100.000000,40.131579
7691,1515915625487742246,100.000000,80.000000
33457,1515915625568676818,100.000000,29.803922
52065,1515915625993327064,99.521531,56.937799


`neutral_share`

In [ ]:
print("Расчет neutral_share...")
neutral_share_columns = apparel_messages.groupby('client_id')['action_type3'].apply(lambda x: (x == 'neutral').sum() / len(x) * 100).reset_index(name='neutral_share')
print(f"Расчет завершен!")
display(neutral_share_columns.sample(5))

Расчет neutral_share...
Расчет завершен!


,client_id,neutral_share
52981,1515915626005953319,43.478261
25958,1515915625536527333,83.536585
37983,1515915625619038421,67.765568
45876,1515915625815460663,49.661400
16587,1515915625490502519,82.068966


`purchase_positive_share`

In [164]:
curr_df = apparel_messages.copy()
curr_df['is_purchase'] = curr_df['event'].apply(lambda x: 1 if x == 'purchase' else 0)
curr_df['is_positive_2'] = curr_df['action_type2'].apply(lambda x: 1 if x == 'positive' else 0)
curr_df['is_positive_3'] = curr_df['action_type3'].apply(lambda x: 1 if x == 'positive' else 0)

purchase_positive_share_columns_3 = curr_df\
    .groupby('client_id')[['is_purchase', 'is_positive_3']]\
    .agg({'is_purchase': 'sum', 'is_positive_3': 'sum'})
    
purchase_positive_share_columns_3['purchase_positive_share_3'] = purchase_positive_share_columns_3\
    .apply(lambda row: (row['is_purchase'] / row['is_positive_3'] * 100) if row['is_purchase'] > 0 else 0, axis=1)
purchase_positive_share_columns_3.drop(columns=['is_purchase', 'is_positive_3'], inplace=True)    
    
purchase_positive_share_columns_2 = curr_df\
    .groupby('client_id')[['is_purchase', 'is_positive_2']]\
    .agg({'is_purchase': 'sum', 'is_positive_2': 'sum'})
    
purchase_positive_share_columns_2['purchase_positive_share_2'] = purchase_positive_share_columns_2\
    .apply(lambda row: (row['is_purchase'] / row['is_positive_2'] * 100) if row['is_purchase'] > 0 else 0, axis=1)
purchase_positive_share_columns_2.drop(columns=['is_purchase', 'is_positive_2'], inplace=True)
    
purchase_positive_share_columns = pd.merge(purchase_positive_share_columns_2, purchase_positive_share_columns_3, on='client_id', how='left')
display(purchase_positive_share_columns.sample(5))

,purchase_positive_share_2,purchase_positive_share_3
client_id,,
1515915625594339866,0.471698,28.571429
1515915625559762766,0.499168,0.955414
1515915625981250118,0.000000,0.000000
1515915625552579299,0.297177,0.699301
1515915625709871274,0.174520,0.386100


`purchase_share`

In [165]:
purchase_share_columns = apparel_messages.groupby('client_id')['event'].apply(lambda x: (x == 'purchase').sum() / len(x) * 100).reset_index(name='purchase_share')
display(purchase_share_columns.sample(5))

,client_id,purchase_share
28655,1515915625557928508,0.840336
47274,1515915625854245159,1.069519
36255,1515915625589217493,0.000000
18429,1515915625491008137,3.240741
35990,1515915625587684917,0.458716


`click_share`

Совпадает с `positive_share_2`

`common_channel`

In [166]:
curr_df = apparel_messages.copy()
common_channel_columns = curr_df.groupby('client_id')['channel'].apply(lambda x: (x == 'email').sum() / len(x) * 100).reset_index(name='common_channel_email')
common_channel_columns['common_channel'] = common_channel_columns['common_channel_email']\
    .apply(lambda x: 'email' if x > 75 else ('mixed' if x > 25 else 'mobile_push'))
common_channel_columns.drop(columns=['common_channel_email'], inplace=True)
display(common_channel_columns.sample(5))

,client_id,common_channel
28335,1515915625557255953,mobile_push
53267,1515915626009387219,mobile_push
46678,1515915625835858071,mobile_push
3114,1515915625470693804,mixed
45223,1515915625800335034,email


####  `full_campaign_daily_event` и `full_campaign_daily_event_channel`

Таблица коэффициентов для действий клиента

| Event | Value |
|-------|-------|
| `open` | 0.1 |
| `click` | 1 |
| `purchase` | 10 |
| `send` | 0 |
| `unsubscribe` | -10 |
| `hbq_spam` | -10 |
| `hard_bounce` | -20 |
| `subscribe` | 5 |
| `soft_bounce` | -5 |
| `complain` | -20 |
| `close` | -5 |

Задаем кооэффициенты

In [167]:
coefs_dict = {
    'open': 0.1,
    'click': 1,
    'purchase': 10,
    'send' : 0,
    'unsubscribe': -20,
    'hbq_spam': -10,
    'hard_bounce': -2,
    'subscribe': 5,
    'soft_bounce': -1,
    'complain': -20,
    'close': -5
}

Для каждого `client_id` и для каждого `bulk_campaign_id` вычисляем значение метрики, для этого:
* Из таблицы `full_campaign_daily_event_channel` для текущего `bulk_campaign_id` берем поле `count` для действия, которое совершил клиент с `client_id` и делим его на сумму всех полей `count` для этой рассылки. Сохарняем эти значения в новое поле таблицы `apparel_messages` 
* Группируем таблицу `apparel_messages` по полю `client_id`, и вычисляем взвешенную сумму значений нового поля для каждого `client_id` (аналогично, можем вычислить сумму только положительных или только отрицательных действий)

Добавляем новое поле

In [ ]:
def get_field_names(event=None, channel='email', aggfunc='nunique'):
    # пример поля: nunique_subscribe_email
    events = ['open', 'click', 'purchase', 'send', 
              'unsubscribe', 'hbq_spam', 'hard_bounce', 
              'subscribe', 'soft_bounce', 'complain', 'close']

    channels = ['email', 'mobile_push']
    
    prohibited_combinations = [('close', 'email'),
                               ('unsubscribe', 'mobile_push'),
                               ('hbq_spam', 'mobile_push'),
                               ('subscribe', 'mobile_push'),
                               ('complain', 'mobile_push')]
    
    field_names = []
    if event is not None:
        event_name = event
        if channel in channels:
            channel_name = channel
            if (event_name, channel_name) not in prohibited_combinations:
                field_names.append(aggfunc + '_' + event_name + '_' + channel_name)
            else:
                print('Prohibited Combination:', event_name, channel_name)
        else:
            for channel_name in channels:
                if (event_name, channel_name) not in prohibited_combinations:
                    field_names.append(aggfunc + '_' + event_name + '_' + channel_name)
    else:
        if channel in channels:
            channel_name = channel
            for event_name in events:
                if (event_name, channel_name) not in prohibited_combinations:
                    field_names.append(aggfunc + '_' + event_name + '_' + channel_name)
        else:
            for channel_name in channels:
                if (event_name, channel_name) not in prohibited_combinations:
                    field_names.append(aggfunc + '_' + event_name + '_' + channel_name)
    return field_names
                
                
        

In [169]:
def compute_new_column(line, columns_numerator, columns_denominator, aggfunc='sum'):
    numerator = 0
    denominator = 0
    for column in columns_numerator:
        numerator += line[column]
    
    for column in columns_denominator:
        denominator += line[column]
        
    if denominator != 0:
        return numerator / denominator
    else:
        return 0

In [ ]:
curr_full_df = pd.merge(apparel_messages, full_campaign_daily_event_channel, on=['bulk_campaign_id', 'date'])
print(f"Merged dataframe shape: {curr_full_df.shape}")
print(f"Total unique bulk_campaign_ids: {curr_full_df['bulk_campaign_id'].nunique()}")
print(f"Total unique created_at: {curr_full_df['created_at'].nunique()}")

# Кэшируем get_field_names для всех уникальных комбинаций event+channel
print("\nПредвычисляем поля для всех комбинаций event+channel...")
start_time = time.time()

# Получаем уникальные комбинации
unique_event_channel = curr_full_df[['event', 'channel']].drop_duplicates()
print(f"Уникальных комбинаций event+channel: {len(unique_event_channel)}")

# Кэшируем результаты get_field_names
field_names_cache = {}
for _, row in unique_event_channel.iterrows():
    event = row['event']
    channel = row['channel']
    key = (event, channel)
    field_names_cache[key] = {
        'numerator': get_field_names(event=event, channel=channel),
        'denominator': get_field_names(event=None, channel=channel)
    }

print(f"Кэш создан!")

# Вычисляем action_ratio с использованием кэша
def compute_action_ratio_cached(row):
    """Оптимизированная версия с кэшем"""
    key = (row['event'], row['channel'])
    num_cols = field_names_cache[key]['numerator']
    denom_cols = field_names_cache[key]['denominator']
    
    numerator = sum(row.get(col, 0) for col in num_cols)
    denominator = sum(row.get(col, 0) for col in denom_cols)
    
    return numerator / denominator if denominator != 0 else 0

print("Обработка 12.7M строк...")
curr_full_df['action_ratio'] = curr_full_df.apply(compute_action_ratio_cached, axis=1)

elapsed_time = time.time() - start_time
print(f"\n Обработка заняла {elapsed_time:.2f} сек ({elapsed_time/60:.2f} мин)")
print(f"Создана колонка action_ratio")
print(f"\nПримеры значений:")
print(curr_full_df[['bulk_campaign_id', 'event', 'channel', 'action_ratio']].head(10))

Merged dataframe shape: (12737931, 43)
Total unique bulk_campaign_ids: 2709
Total unique created_at: 4102608

Предвычисляем поля для всех комбинаций event+channel...
Уникальных комбинаций event+channel: 17
Кэш создан!
Обработка 12.7M строк...

 Обработка заняла 224.82 сек (3.75 мин)
Создана колонка action_ratio

Примеры значений:
   bulk_campaign_id  event channel  action_ratio
0              4439   open   email      0.774237
1              4439   open   email      0.774237
2              4439   open   email      0.774237
3              4439  click   email      0.184581
4              4439   open   email      0.774237
5              4439   open   email      0.774237
6              4439   open   email      0.774237
7              4439   open   email      0.774237
8              4439  click   email      0.184581
9              4439   open   email      0.774237


In [ ]:
print("Создание новых признаков action_cost и action_coef...")
print(f"Всего строк для обработки: {len(curr_full_df):,}\n")

start_time = time.time()

curr_full_df = curr_full_df[['bulk_campaign_id', 'client_id', 'date', 'event', 'channel', 'action_ratio']]

# Вычисляем коэффициенты для каждого события
print("Вычисление коэффициентов событий...")
coef_values = curr_full_df['event'].map(coefs_dict).fillna(0)

# Вычисляем action_cost
print("Вычисление action_cost...")
curr_full_df['action_cost'] = coef_values * np.exp(-curr_full_df['action_ratio'])

# коэффициент события
print("Вычисление action_coef...")
curr_full_df['action_coef'] = coef_values

elapsed_time = time.time() - start_time
print(f"\nОбработка заняла {elapsed_time:.2f} сек ({elapsed_time/60:.2f} мин)")

Создание новых признаков action_cost и action_coef...
Всего строк для обработки: 12,737,931

Вычисление коэффициентов событий...
Вычисление action_cost...
Вычисление action_coef...

Обработка заняла 1.42 сек (0.02 мин)


Проверка

In [173]:
curr_full_df.head()

,bulk_campaign_id,client_id,date,event,channel,action_ratio,action_cost,action_coef
0,4439,1515915625626736623,2022-05-19,open,email,0.774237,0.046106,0.1
1,4439,1515915625490086521,2022-05-19,open,email,0.774237,0.046106,0.1
2,4439,1515915625553578558,2022-05-19,open,email,0.774237,0.046106,0.1
3,4439,1515915625553578558,2022-05-19,click,email,0.184581,0.831452,1.0
4,4439,1515915625471518311,2022-05-19,open,email,0.774237,0.046106,0.1


`weighted_effect`

In [174]:
weighted_effect_columns = (curr_full_df
                           .groupby('client_id')[['action_ratio', 'action_cost']]
                           .agg({'action_ratio': 'mean', 'action_cost': 'sum'})
                           .rename(columns={'action_ratio': 'avg_actions_ratio', 'action_cost': 'sum_actions_cost'})
                           .reset_index(drop=False))
display(weighted_effect_columns)


,client_id,avg_actions_ratio,sum_actions_cost
0,1515915625468060902,0.759644,42.115687
1,1515915625468061003,0.877879,16.103515
2,1515915625468061099,0.730997,7.957753
3,1515915625468061100,0.541356,19.585162
4,1515915625468061170,0.764067,46.096103
...,...,...,...
53324,1515915626010183608,0.490580,0.138314
53325,1515915626010221592,0.291606,3.060420
53326,1515915626010234726,0.985885,0.000000
53327,1515915626010261344,0.331907,1.068075


`weighted_effect_positive`

In [175]:
weighted_effect_positive_columns = (curr_full_df[curr_full_df['action_coef'] > 0]
                           .groupby('client_id')[['action_ratio', 'action_cost']]
                           .agg({'action_ratio': 'mean', 'action_cost': 'sum'})
                           .rename(columns={'action_ratio': 'avg_positive_actions_ratio', 'action_cost': 'sum_positive_actions_cost'})
                           .reset_index(drop=False))

display(weighted_effect_positive_columns)

,client_id,avg_positive_actions_ratio,sum_positive_actions_cost
0,1515915625468060902,0.345211,62.103976
1,1515915625468061003,0.176781,16.103515
2,1515915625468061099,0.247532,11.876227
3,1515915625468061100,0.337602,22.565396
4,1515915625468061170,0.378545,46.096103
...,...,...,...
52073,1515915626010172489,0.368906,0.069149
52074,1515915626010183608,0.368835,0.138314
52075,1515915626010221592,0.163319,3.060420
52076,1515915626010261344,0.189284,1.068075


`weighted_effect_negative`

In [176]:
weighted_effect_negative_columns = (curr_full_df[curr_full_df['action_coef'] < 0]
                           .groupby('client_id')[['action_ratio', 'action_cost']]
                           .agg({'action_ratio': 'mean', 'action_cost': 'sum'})
                           .rename(columns={'action_ratio': 'avg_negative_actions_ratio', 'action_cost': 'sum_negative_actions_cost'})
                           .reset_index(drop=False))

display(weighted_effect_negative_columns)

,client_id,avg_negative_actions_ratio,sum_negative_actions_cost
0,1515915625468060902,0.000586,-19.988289
1,1515915625468061099,0.020659,-3.918475
2,1515915625468061100,0.005408,-2.980235
3,1515915625468061975,0.029087,-9.713317
4,1515915625468062558,0.009032,-1.982017
...,...,...,...
16258,1515915626009289751,0.005742,-1.988550
16259,1515915626009370761,0.005742,-1.988550
16260,1515915626009421834,0.003562,-1.992889
16261,1515915626009502220,0.004358,-1.991303


### Полученные датафреймы:

#### Apparel_purchases:
* `total_quatity_columns` - столбцы с общим числом покупок за период для каждого пользователя

* `total_spent_columns` - столбцы с общим числом потраченных средств за период для каждого пользователя

* `avg_quantity_columns` -   -//- средним числом покупок                               -//-

* `avg_quantity_per_order_columns` - столбец со средним числом покупок за заказ

* `avg_spent_columns_` - столбец со средним чеком за заказ

* `best_categories_level_columns` - столбцы с лучшими категориями на каждом из уровней, их количество и процент появления для каждого клиента

* `days_after_last_purchase` - столбец с разницей между максимальной датой рассылки в таблице `apparel_purchases` и максимальной датой покупки для каждого клиента

#### Apparel_messages:

* `total_messages_columns` - столбец с числом рассылок, отправленных клиенту

* `positive_share_columns` - столбцы с долей положительных действий для двух вариантов группировки действий

* `neutral_share_columns` - столбец с долей нейтральных действий для второго варианта группировки действий, т.к. в первом варианте есть только положительные и отрицательные действия

* `purchase_positive_share_columns` - столбцы с долей покупок от доли положительных действий для двух вариантов группировки действий

* `purchase_share_columns` - столбец с долей покупок от всех действий клиента

* `common_channel_columns` - столбец с наиболее популярным способом получения рассылок

#### Full_campaign_daily_event(_channel)

* `weighted_effect_columns` - столбец со средней долей, которую действия клиентов составляют от числа всех действий (т.е. то, насколько действие клиента типично), а также столбец с суммарной 'стоимостью' действий клиента

* `weighted_effect_positive_columns` - столбец со средней долей, которую *положительные* действия клиентов составляют от числа всех действий (т.е. то, насколько действие клиента типично), а также столбец с суммарной 'стоимостью' *положительных* действий клиента

* `weighted_effect_negative_columns` - столбец со средней долей, которую *отрицательные* действия клиентов составляют от числа всех действий (т.е. то, насколько действие клиента типично), а также столбец с суммарной 'стоимостью' *отрицательных* действий клиента





#### Объединяем:

`Apparel_purchases`

In [177]:
#apparel_purchases_features = target_table.drop(columns=['target'])
purchases_features_dataframes = [total_quantity_columns,
                                 total_spent_columns,
                                 avg_quantity_columns,
                                 avg_quantity_per_order_columns,
                                 avg_spent_columns,
                                 best_categories_level_columns,
                                 days_after_last_purchase_columns]

for i, df_to_merge in enumerate(purchases_features_dataframes):
    if i == 0:
        apparel_purchases_features = df_to_merge
    else:
        apparel_purchases_features = pd.merge(apparel_purchases_features, df_to_merge, on='client_id', how='left')
    
display(apparel_purchases_features.head())


,client_id,total_quantity_week,total_quantity_month,total_quantity_3_month,total_quantity_half_year,total_quantity_year,total_quantity_3_years,total_spent_week,total_spent_month,total_spent_3_month,...,best_category_level_2,best_count_level_2,best_percentage_level_2,best_category_level_3,best_count_level_3,best_percentage_level_3,best_category_level_4,best_count_level_4,best_percentage_level_4,days_after_last_purchase
0,1515915625468060902,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,28,4.0,57.142857,260,2.0,28.571429,420,2.0,28.571429,630
1,1515915625468061003,0.0,0.0,0.0,0.0,0.0,7,0.0,0.0,0.0,...,28,7.0,100.000000,249,7.0,100.000000,615,7.0,100.000000,408
2,1515915625468061099,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,...,28,1.0,100.000000,290,1.0,100.000000,424,1.0,100.000000,640
3,1515915625468061100,2.0,2.0,2.0,2.0,2.0,2,2098.0,2098.0,2098.0,...,27,2.0,100.000000,1828,2.0,100.000000,5717,2.0,100.000000,6
4,1515915625468061170,0.0,0.0,0.0,0.0,19.0,19,0.0,0.0,0.0,...,28,15.0,88.235294,260,12.0,70.588235,420,12.0,70.588235,244


`Apparel_messages`

In [178]:
#apparel_messages_features = target_table.drop(columns=['target'])
messages_features_dataframes = [total_messages_columns,
                                positive_share_columns,
                                neutral_share_columns,
                                purchase_positive_share_columns,
                                purchase_share_columns,
                                common_channel_columns]


for i, df_to_merge in enumerate(messages_features_dataframes):
    if i == 0:
        apparel_messages_features = df_to_merge
    else:
        apparel_messages_features = pd.merge(apparel_messages_features, df_to_merge, on='client_id', how='left')
    
display(apparel_messages_features.head())

,client_id,total_messages,positive_share_2,positive_share_3,neutral_share,purchase_positive_share_2,purchase_positive_share_3,purchase_share,common_channel
0,1515915625468060902,177,99.435028,28.248588,71.186441,2.840909,10.000000,2.824859,email
1,1515915625468061003,166,100.000000,7.228916,92.771084,0.602410,8.333333,0.602410,email
2,1515915625468061099,276,99.275362,21.376812,77.898551,0.000000,0.000000,0.000000,mixed
3,1515915625468061100,434,99.539171,38.018433,61.520737,0.231481,0.606061,0.230415,mobile_push
4,1515915625468061170,293,100.000000,17.064846,82.935154,1.023891,6.000000,1.023891,mixed


`Full_campaign_daily_event(_channel)`

In [179]:
#full_campaign_daily_event_features = target_table.drop(columns=['target'])
aggregate_features_dataframes = [weighted_effect_columns,
                                 weighted_effect_positive_columns,
                                 weighted_effect_negative_columns]


for i, df_to_merge in enumerate(aggregate_features_dataframes):
    if i == 0:
        full_campaign_daily_event_features = df_to_merge
    else:
        full_campaign_daily_event_features = pd.merge(full_campaign_daily_event_features, df_to_merge, on='client_id', how='left')
    
display(full_campaign_daily_event_features.head())

,client_id,avg_actions_ratio,sum_actions_cost,avg_positive_actions_ratio,sum_positive_actions_cost,avg_negative_actions_ratio,sum_negative_actions_cost
0,1515915625468060902,0.759644,42.115687,0.345211,62.103976,0.000586,-19.988289
1,1515915625468061003,0.877879,16.103515,0.176781,16.103515,NaN,NaN
2,1515915625468061099,0.730997,7.957753,0.247532,11.876227,0.020659,-3.918475
3,1515915625468061100,0.541356,19.585162,0.337602,22.565396,0.005408,-2.980235
4,1515915625468061170,0.764067,46.096103,0.378545,46.096103,NaN,NaN


Итоговый датафрейм

In [180]:
final_df = target_table.drop(columns=['target'])

final_df = pd.merge(final_df, apparel_purchases_features, on='client_id', how='left')
final_df = pd.merge(final_df, apparel_messages_features, on='client_id', how='left')
final_df = pd.merge(final_df, full_campaign_daily_event_features, on='client_id', how='left')

Сохранение полученных датафреймов

In [181]:
apparel_purchases_features.to_csv('./Данные/apparel_purchases_features.csv', index=False, sep=";")
apparel_messages_features.to_csv('./Данные/apparel_messages_features.csv', index=False, sep=";")
full_campaign_daily_event_features.to_csv('./Данные/full_campaign_daily_event_features.csv', index=False, sep=";")
final_df.to_csv('./Данные/model_features.csv', index=False, sep=";")